# RAG with Source Citations

An enhanced RAG pipeline that builds on `simple_rag.ipynb` and adds:
- **Source tracking** with citations in the response
- **Relevance/confidence scores** for each retrieved chunk
- A **reranking step** to improve retrieval quality
- **Formatted output** with numbered references

This demonstrates a more production-ready approach to RAG.

**Prerequisites:** Make sure you have your `ANTHROPIC_API_KEY` set as an environment variable.

**Note:** This notebook requires the `DOCUMENTS` data from `sample_documents.ipynb`. You can either:
1. Run `sample_documents.ipynb` first and copy the `DOCUMENTS` variable, or
2. The `DOCUMENTS` data is imported from `sample_documents` below.

## Install Dependencies

In [ ]:
!pip install anthropic chromadb

## Imports

In [ ]:
import os
import textwrap
from dataclasses import dataclass

import anthropic
import chromadb

## Load Sample Documents

Import the sample knowledge base from `sample_documents`.

**Option A:** If you have already run `sample_documents.ipynb`, you can import from there.

**Option B:** Copy the `DOCUMENTS` list directly from `sample_documents.ipynb` into this notebook for standalone use.

In [ ]:
from sample_documents import DOCUMENTS

## Data Structures

We use a dataclass to keep retrieved chunks organized with their metadata, distance scores, and reranking scores.

In [ ]:
@dataclass
class RetrievedChunk:
    """Represents a single retrieved chunk with all its metadata."""

    text: str
    source_title: str
    category: str
    chunk_index: int
    distance: float  # Raw distance from vector search (lower = more similar)
    relevance_score: float  # Normalized relevance score (0-1, higher = better)
    rerank_score: float = 0.0  # Score after reranking

## Chunking -- Improved Boundary Detection

Same concept as `simple_rag.ipynb` but with better boundary detection -- we try to split at paragraph boundaries when possible instead of cutting mid-sentence.

In [ ]:
def chunk_document(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Split a document into overlapping chunks, trying to break at paragraph
    boundaries when possible.
    """
    # First, try splitting on double newlines (paragraph breaks)
    paragraphs = text.split("\n\n")

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:
        # If adding this paragraph would exceed the chunk size, save current and start new
        if len(current_chunk) + len(paragraph) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Keep the last `overlap` characters for context continuity
            current_chunk = current_chunk[-overlap:] + "\n\n" + paragraph
        else:
            if current_chunk:
                current_chunk += "\n\n" + paragraph
            else:
                current_chunk = paragraph

    # Don't forget the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

### Prepare Chunks for ChromaDB

Chunk all documents and prepare the parallel lists for ChromaDB. This version includes additional metadata fields (department, last_updated) for richer source tracking.

In [ ]:
def prepare_chunks(documents: list[dict]) -> tuple[list[str], list[str], list[dict]]:
    """Chunk all documents and prepare parallel lists for ChromaDB."""
    ids = []
    texts = []
    metadatas = []

    for doc_idx, doc in enumerate(documents):
        chunks = chunk_document(doc["content"])
        for chunk_idx, chunk_text in enumerate(chunks):
            chunk_id = f"doc{doc_idx}_chunk{chunk_idx}"
            ids.append(chunk_id)
            texts.append(chunk_text)
            metadatas.append(
                {
                    "source_title": doc["title"],
                    "category": doc["metadata"]["category"],
                    "department": doc["metadata"].get("department", "unknown"),
                    "last_updated": doc["metadata"].get("last_updated", "unknown"),
                    "chunk_index": chunk_idx,
                }
            )

    return ids, texts, metadatas

## Vector Store

Create a ChromaDB collection and load all chunks. Same approach as the simple version.

In [ ]:
def build_vector_store(
    ids: list[str], texts: list[str], metadatas: list[dict]
) -> chromadb.Collection:
    """Create a ChromaDB collection and load all chunks."""
    client = chromadb.Client()
    collection = client.create_collection(
        name="acme_knowledge_base_enhanced",
        metadata={"description": "Acme Corp knowledge base with enhanced retrieval"},
    )
    collection.add(ids=ids, documents=texts, metadatas=metadatas)
    print(f"  Indexed {collection.count()} chunks.")
    return collection

## Retrieval with Scoring

We retrieve more candidates than we need, then rerank to find the best ones.

ChromaDB returns distances (lower = more similar). We convert these to relevance scores (higher = more relevant) on a 0-1 scale using `1 / (1 + distance)`.

In [ ]:
def retrieve_with_scores(
    collection: chromadb.Collection, query: str, top_k: int = 10
) -> list[RetrievedChunk]:
    """
    Retrieve chunks from the vector store and compute relevance scores.

    We fetch `top_k` candidates (more than we'll ultimately use) so the
    reranker has a larger pool to work with.

    ChromaDB returns distances (lower = more similar). We convert these
    to relevance scores (higher = more relevant) on a 0-1 scale.
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    for i in range(len(results["documents"][0])):
        distance = results["distances"][0][i]

        # Convert distance to a relevance score between 0 and 1.
        # ChromaDB uses L2 (Euclidean) distance by default.
        # Smaller distance = more similar, so we invert it.
        # We use 1 / (1 + distance) as a simple normalization.
        relevance_score = 1.0 / (1.0 + distance)

        chunk = RetrievedChunk(
            text=results["documents"][0][i],
            source_title=results["metadatas"][0][i]["source_title"],
            category=results["metadatas"][0][i]["category"],
            chunk_index=results["metadatas"][0][i]["chunk_index"],
            distance=distance,
            relevance_score=relevance_score,
        )
        chunks.append(chunk)

    return chunks

## Reranking

Reranking is a second pass that scores retrieved chunks more carefully.

In production, you'd use a dedicated cross-encoder reranking model (e.g., Cohere Rerank, or a cross-encoder from sentence-transformers).

Here we implement a simple keyword + heuristic reranker to demonstrate the concept without adding another dependency. Our reranking score combines:
1. The original vector similarity score (semantic match)
2. Keyword overlap (exact word match between query and chunk)
3. Freshness bonus (recently updated documents get a slight boost)

In [ ]:
def rerank(query: str, chunks: list[RetrievedChunk], top_k: int = 5) -> list[RetrievedChunk]:
    """
    Rerank retrieved chunks using a simple scoring heuristic.

    Our reranking score combines:
      1. The original vector similarity score (semantic match)
      2. Keyword overlap (exact word match between query and chunk)
      3. Freshness bonus (recently updated documents get a slight boost)

    In production, replace this with a proper cross-encoder model.
    """
    query_words = set(query.lower().split())

    for chunk in chunks:
        # --- Factor 1: Original relevance score (0.0 to 1.0) ---
        semantic_score = chunk.relevance_score

        # --- Factor 2: Keyword overlap score ---
        # Count what fraction of query words appear in the chunk text
        chunk_words = set(chunk.text.lower().split())
        if query_words:
            keyword_overlap = len(query_words & chunk_words) / len(query_words)
        else:
            keyword_overlap = 0.0

        # --- Factor 3: Freshness bonus ---
        # Give a small bonus to recently updated documents
        freshness_bonus = 0.0
        # (In a real system, you'd compare dates. Here we just give a tiny
        #  static bonus to show the concept.)

        # --- Combine scores ---
        # Weight: 60% semantic, 30% keyword, 10% freshness
        chunk.rerank_score = (
            0.6 * semantic_score + 0.3 * keyword_overlap + 0.1 * freshness_bonus
        )

    # Sort by rerank score (highest first) and return top_k
    chunks.sort(key=lambda c: c.rerank_score, reverse=True)
    return chunks[:top_k]

## Prompt Construction with Citation Instructions

Each retrieved chunk is labeled with a reference number `[1]`, `[2]`, etc. The LLM is instructed to include these reference numbers in its answer so the user can trace claims back to specific documents.

In [ ]:
def build_prompt_with_citations(query: str, chunks: list[RetrievedChunk]) -> str:
    """
    Build an augmented prompt that instructs the LLM to cite sources.

    Each retrieved chunk is labeled with a reference number [1], [2], etc.
    The LLM is instructed to include these reference numbers in its answer
    so the user can trace claims back to specific documents.
    """
    context_sections = []
    for i, chunk in enumerate(chunks, 1):
        section = (
            f"[{i}] Source: {chunk.source_title} "
            f"(Category: {chunk.category}, Relevance: {chunk.rerank_score:.2f})\n"
            f"{chunk.text}"
        )
        context_sections.append(section)

    context_block = "\n\n---\n\n".join(context_sections)

    prompt = (
        "You are a knowledgeable assistant for Acme Corp. Answer the user's "
        "question based ONLY on the provided context documents.\n\n"
        "IMPORTANT RULES:\n"
        "1. Only use information from the provided context.\n"
        "2. Cite your sources using reference numbers like [1], [2], etc.\n"
        "3. If the context doesn't contain enough information, say so.\n"
        "4. Be concise but thorough.\n\n"
        f"CONTEXT DOCUMENTS:\n\n{context_block}\n\n"
        f"QUESTION: {query}\n\n"
        "Provide your answer with citations:"
    )
    return prompt

## LLM Response Generation

Send the augmented prompt to Claude and return the response.

In [ ]:
def generate_response(prompt: str) -> str:
    """Send the augmented prompt to Claude and return the response."""
    client = anthropic.Anthropic(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
    )

    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return message.content[0].text

## Output Formatting

Format the final output with the answer, source list, relevance bars, and an overall confidence assessment.

In [ ]:
def format_response(
    question: str, answer: str, chunks: list[RetrievedChunk]
) -> str:
    """
    Format the final output with the answer, source list, and confidence info.
    """
    separator = "=" * 70
    thin_sep = "-" * 70

    output_parts = [
        separator,
        f"QUESTION: {question}",
        separator,
        "",
        "ANSWER:",
        "",
    ]

    # Wrap the answer text for readability
    for line in answer.split("\n"):
        output_parts.append(textwrap.fill(line, width=70))

    output_parts.extend(["", thin_sep, "SOURCES:", thin_sep, ""])

    # List each source with its relevance info
    for i, chunk in enumerate(chunks, 1):
        score_bar = "#" * int(chunk.rerank_score * 20)
        score_bar = score_bar.ljust(20, ".")
        output_parts.append(
            f"  [{i}] {chunk.source_title}\n"
            f"      Category:  {chunk.category}\n"
            f"      Relevance: [{score_bar}] {chunk.rerank_score:.2%}\n"
        )

    # Overall confidence assessment
    avg_score = sum(c.rerank_score for c in chunks) / len(chunks) if chunks else 0
    if avg_score > 0.4:
        confidence = "HIGH -- Retrieved documents are highly relevant"
    elif avg_score > 0.25:
        confidence = "MEDIUM -- Some relevant documents found"
    else:
        confidence = "LOW -- Retrieved documents may not be relevant"

    output_parts.extend(
        [
            thin_sep,
            f"CONFIDENCE: {confidence} (avg relevance: {avg_score:.2%})",
            separator,
        ]
    )

    return "\n".join(output_parts)

## Main Pipeline

Run the full enhanced RAG pipeline:

**Chunk -> Index -> Retrieve -> Rerank -> Augment -> Generate -> Format**

In [ ]:
def run_enhanced_rag(question: str, verbose: bool = True) -> str:
    """
    Run the full enhanced RAG pipeline:
      Chunk -> Index -> Retrieve -> Rerank -> Augment -> Generate -> Format

    Args:
        question: The user's natural language question.
        verbose:  If True, print progress messages.

    Returns:
        The formatted response string.
    """
    if verbose:
        print("\n[1/5] Chunking and indexing documents...")

    ids, texts, metadatas = prepare_chunks(DOCUMENTS)
    collection = build_vector_store(ids, texts, metadatas)

    if verbose:
        print(f"[2/5] Retrieving candidates for: '{question}'")

    # Retrieve more candidates than we need — the reranker will narrow it down
    candidates = retrieve_with_scores(collection, question, top_k=10)

    if verbose:
        print(f"  Retrieved {len(candidates)} candidates from vector search.")
        print("[3/5] Reranking candidates...")

    # Rerank and keep the top 5
    top_chunks = rerank(question, candidates, top_k=5)

    if verbose:
        print(f"  Top {len(top_chunks)} chunks after reranking:")
        for i, chunk in enumerate(top_chunks, 1):
            print(f"    {i}. {chunk.source_title} (score: {chunk.rerank_score:.3f})")
        print("[4/5] Building augmented prompt with citation instructions...")

    prompt = build_prompt_with_citations(question, top_chunks)

    if verbose:
        print("[5/5] Generating response with Claude...")

    answer = generate_response(prompt)

    # Format the final output
    formatted = format_response(question, answer, top_chunks)
    return formatted

## Run the Enhanced Pipeline

Check for the API key, then run the enhanced RAG pipeline on example questions that demonstrate different retrieval scenarios.

In [ ]:
# Check for API key
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ERROR: Please set the ANTHROPIC_API_KEY environment variable.")
    print("  export ANTHROPIC_API_KEY='your-key-here'")
else:
    print("API key found. Ready to run the enhanced pipeline.")

In [ ]:
# Example questions that show off different retrieval scenarios
questions = [
    "What is the return policy for electronics and how long do refunds take?",
    "How much PTO do I get if I've been at the company for 4 years?",
    "Tell me about the SmartHome Hub Pro — what protocols does it support?",
    "What are the benefits of being a Platinum rewards member?",
    "Can I work from home on Mondays?",
]

In [ ]:
# Run the enhanced pipeline for each question
for question in questions:
    result = run_enhanced_rag(question)
    print(result)
    print("\n")